# Notebook 01 — Foco Antioquia: Deslizamientos (target binario)

**Objetivo:** Acotar el experimento al departamento de Antioquia, filtrar
exclusivamente eventos de remoción en masa, y construir el **target binario**
(ocurre / no ocurre deslizamiento) como base para la fase de modelado.

**Configuración:** `configs/antioquia_deslizamientos.yaml`
**Continuación de:** `00_experimento_correlacion.ipynb`

> **Cambio respecto a la versión anterior:** El target pasó de multiclase
> (bajo/medio/alto) a **binario**. La razón es que los umbrales del modelo
> multiclase eran arbitrarios. 

## 0. Entorno y configuración

In [ ]:
import sys
from pathlib import Path

_cwd = Path().resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / "pyproject.toml").exists():
        _root = _p
        break
else:
    _root = _cwd.parent

sys.path.insert(0, str(_root / "src"))

import warnings
warnings.filterwarnings("ignore")

import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from experiment.config import load_config
from experiment.download import load_ideam, load_ungrd
from experiment.process import clean_ideam

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid")

cfg = load_config(project_root=_root)

print("✓ Configuración cargada")
print(f"  Departamento  : {cfg.geo.departamento}")
print(f"  Período       : {cfg.periodo.anio_inicio} – {cfg.periodo.anio_fin}")
print(f"  Target        : {cfg.target.tipo} — '{cfg.target.nombre}'")
print(f"  Clase positiva: {cfg.target.clase_positiva} ({cfg.target.clase_positiva_desc})")
print(f"  Clase negativa: {cfg.target.clase_negativa} ({cfg.target.clase_negativa_desc})")
print(f"  class_weight  : {cfg.target.class_weight}")
print(f"  Features      : {len(cfg.all_features)}")

## 1. Carga de datos crudos

In [ ]:
df_ideam_raw = load_ideam(
    anio_inicio=cfg.periodo.anio_inicio,
    anio_fin=cfg.periodo.anio_fin,
)
df_ungrd_raw = load_ungrd()

print(f"IDEAM crudo : {df_ideam_raw.shape}")
print(f"UNGRD crudo : {df_ungrd_raw.shape}")
print(f"\nColumnas UNGRD: {list(df_ungrd_raw.columns)}")

## 2. Limpieza y filtro geográfico — Antioquia

In [ ]:
def _normalize(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFD", s.lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

df_ideam_clean = clean_ideam(df_ideam_raw)

df_ideam = df_ideam_clean[
    df_ideam_clean["departamento"].apply(_normalize) == cfg.geo.departamento
].copy()

print(f"IDEAM Colombia  : {len(df_ideam_clean):,} registros")
print(f"IDEAM Antioquia : {len(df_ideam):,} ({len(df_ideam)/len(df_ideam_clean):.1%})")
print(f"Rango temporal  : {df_ideam['fecha'].min().date()} → {df_ideam['fecha'].max().date()}")

## 3. Filtro UNGRD — solo deslizamientos

In [ ]:
evento_col = next(
    (c for c in df_ungrd_raw.columns if any(k in c.lower() for k in ["evento", "tipo"])),
    None,
)
dpto_col = next((c for c in df_ungrd_raw.columns if "depart" in c.lower()), None)
date_col = next((c for c in df_ungrd_raw.columns if "fecha"  in c.lower()), None)

df_ungrd = df_ungrd_raw.copy()

if evento_col:
    mask_evento = df_ungrd[evento_col].apply(
        lambda v: any(k in _normalize(str(v)) for k in cfg.eventos.landslide_keywords)
    )
    df_ungrd = df_ungrd[mask_evento].copy()
    print(f"Después del filtro de evento : {len(df_ungrd):,}")

if dpto_col:
    mask_dpto = df_ungrd[dpto_col].apply(lambda v: _normalize(str(v)) == cfg.geo.departamento)
    df_ungrd = df_ungrd[mask_dpto].copy()
    print(f"Después del filtro geográfico: {len(df_ungrd):,}")

df_ungrd["fecha"]       = pd.to_datetime(df_ungrd[date_col], errors="coerce")
df_ungrd                = df_ungrd.dropna(subset=["fecha"])
df_ungrd["anio"]        = df_ungrd["fecha"].dt.year
df_ungrd["mes"]         = df_ungrd["fecha"].dt.month
df_ungrd["anio_mes"]    = df_ungrd["fecha"].dt.to_period("M")
df_ungrd["tipo_evento"] = df_ungrd[evento_col] if evento_col else "deslizamiento"

print(f"\n✓ UNGRD Antioquia (deslizamientos): {len(df_ungrd):,} eventos")
print(f"  Período: {df_ungrd['fecha'].min().date()} → {df_ungrd['fecha'].max().date()}")

## 4. Agregación mensual y construcción del target binario

> **¿Por qué binario?**
> El target `deslizamiento = 1` si ocurrió al menos un evento ese mes,
> `0` si no ocurrió ninguno. No necesita umbrales subjetivos.
> El desbalance de clases se maneja con `class_weight` en el modelo,
> no manipulando el target.

In [ ]:
# Precipitación mensual
precip_monthly = (
    df_ideam
    .groupby("anio_mes", observed=True)
    .agg(
        precip_suma  = ("precip_mm", "sum"),
        precip_max   = ("precip_mm", "max"),
        precip_media = ("precip_mm", "mean"),
        precip_dias  = ("precip_mm", lambda x: (x > 0).sum()),
        n_estaciones = ("precip_mm", "count"),
    )
    .reset_index()
)

# Conteo de deslizamientos por mes
desli_monthly = (
    df_ungrd
    .groupby("anio_mes", observed=True)
    .size()
    .reset_index(name="n_deslizamientos")
)

# Join y target binario
df_monthly = precip_monthly.merge(desli_monthly, on="anio_mes", how="left")
df_monthly["n_deslizamientos"] = df_monthly["n_deslizamientos"].fillna(0).astype(int)
df_monthly["anio"] = df_monthly["anio_mes"].dt.year
df_monthly["mes"]  = df_monthly["anio_mes"].dt.month

# Target: 1 si ocurrió al menos un deslizamiento, 0 si no
# No hay umbral arbitrario — la definición es exactamente esta
df_monthly[cfg.target.nombre] = (df_monthly["n_deslizamientos"] >= 1).astype(int)
df_monthly = df_monthly.sort_values("anio_mes").reset_index(drop=True)

print(f"Dataset mensual: {len(df_monthly):,} meses")
print(f"\nBalance del target '{cfg.target.nombre}':")
vc = df_monthly[cfg.target.nombre].value_counts().sort_index()
for clase, n in vc.items():
    desc = cfg.target.clase_positiva_desc if clase == 1 else cfg.target.clase_negativa_desc
    print(f"  {clase} ({desc}): {n} meses ({n/len(df_monthly):.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Balance de clases
colores = {0: "#2ecc71", 1: "#e74c3c"}
vc = df_monthly[cfg.target.nombre].value_counts().sort_index()
labels = [cfg.target.clase_negativa_desc, cfg.target.clase_positiva_desc]
axes[0].bar(
    [f"{k}\n{labels[k]}" for k in vc.index],
    vc.values,
    color=[colores[k] for k in vc.index],
    edgecolor="white",
    width=0.5,
)
axes[0].set_title(f"Balance del target binario\n('{cfg.target.nombre}')")
axes[0].set_ylabel("Meses")
for i, val in enumerate(vc.values):
    axes[0].text(i, val + 0.2, str(val), ha="center", fontsize=11)

# Serie temporal
ts = df_monthly.copy()
ts.index = ts["anio_mes"].dt.to_timestamp()
axes[1].bar(ts.index, ts["n_deslizamientos"], color="#c0392b", width=20, alpha=0.8, edgecolor="white")
axes[1].axhline(1, color="#e74c3c", linestyle="--", linewidth=1.2, label="Umbral binario (≥1)")
axes[1].set_title(f"Deslizamientos mensuales — {cfg.geo.departamento.title()}")
axes[1].set_ylabel("N° eventos")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Feature engineering

In [ ]:
df = df_monthly.sort_values("anio_mes").copy()

df["precip_lag1"]   = df["precip_suma"].shift(1)
df["precip_acum3m"] = df["precip_suma"].shift(1).rolling(3, min_periods=1).sum()
df["precip_acum6m"] = df["precip_suma"].shift(1).rolling(6, min_periods=1).sum()
df["precip_max3m"]  = df["precip_max"].shift(1).rolling(3,  min_periods=1).max()
df["precip_dias3m"] = df["precip_dias"].shift(1).rolling(3, min_periods=1).sum()
df["mes_sin"]       = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"]       = np.cos(2 * np.pi * df["mes"] / 12)

df = df.dropna(subset=cfg.features.required_for_model).reset_index(drop=True)
FEATURES = [f for f in cfg.all_features if f in df.columns]

print(f"Dataset con features: {len(df):,} meses")
print(f"Features activas ({len(FEATURES)}): {FEATURES}")
display(df[FEATURES + ["n_deslizamientos", cfg.target.nombre]].describe().round(2))

## 6. Calidad de datos — cobertura de sensores

In [ ]:
umbral_cobertura = df["n_estaciones"].quantile(cfg.calidad.cobertura_sensor_percentil / 100)

missings = df[FEATURES].isnull().sum()
diag = missings[missings > 0].to_frame("n_missing")
diag["pct"] = (diag["n_missing"] / len(df) * 100).round(1)

print("=" * 50)
print("DIAGNÓSTICO DE DATOS FALTANTES")
print("=" * 50)
if len(diag):
    display(diag)
else:
    print("✓ Sin datos faltantes.")

meses_baja = df[df["n_estaciones"] < umbral_cobertura]
print(f"\nMeses con baja cobertura (<p{cfg.calidad.cobertura_sensor_percentil}): {len(meses_baja)}")

## 7. Persistencia y tracking MLflow

In [ ]:
import shutil

cfg.processed_dir.mkdir(parents=True, exist_ok=True)
output_path = cfg.processed_dir / f"{cfg.geo.departamento}_deslizamientos_binario_v1.csv"
df.to_csv(output_path, index=False)

config_snapshot = cfg.processed_dir / "config_snapshot_v2.yaml"
shutil.copy(_root / "configs" / "antioquia_deslizamientos.yaml", config_snapshot)

print(f"✓ Dataset: {output_path.relative_to(_root)}")
print(f"✓ Config snapshot: {config_snapshot.name}")

In [ ]:
mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name="dataset_binario_v1", tags=cfg.mlflow.run_tags) as run:
    mlflow.log_params(cfg.as_mlflow_params())

    n_pos = int(df[cfg.target.nombre].sum())
    n_neg = len(df) - n_pos

    mlflow.log_metrics({
        "n_meses"              : len(df),
        "n_deslizamientos"     : int(df["n_deslizamientos"].sum()),
        "n_meses_con_evento"   : n_pos,
        "n_meses_sin_evento"   : n_neg,
        "pct_clase_positiva"   : float(n_pos / len(df)),
        "ratio_desbalance"     : float(n_neg / n_pos) if n_pos > 0 else 0,
        "precip_media_mensual" : float(df["precip_suma"].mean()),
        "n_features"           : len(FEATURES),
    })

    mlflow.log_artifact(str(output_path))
    mlflow.log_artifact(str(config_snapshot))

    print(f"✓ MLflow run: {run.info.run_id}")
    print(f"  Experimento: {cfg.mlflow.experiment_name}")

## 8. Resumen y entregable para Notebook 02

El dataset resultante tiene la siguiente estructura lista para modelado:

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `anio_mes` | Period | Período mensual |
| `precip_suma` … `precip_dias3m` | float | Features de precipitación |
| `mes_sin`, `mes_cos` | float | Estacionalidad cíclica |
| `n_deslizamientos` | int | Conteo real (no se usa en modelo, solo para análisis) |
| `deslizamiento` | int (0/1) | **Target binario** |

**Notebook 02** tomará este CSV y construirá la fase de experiment tracking:
evaluación de múltiples algoritmos, validación cruzada y registro en MLflow.

In [ ]:
print("=" * 60)
print("RESUMEN — DATASET BINARIO v1")
print("=" * 60)
print(f"  Departamento   : {cfg.geo.departamento.title()}")
print(f"  Período        : {cfg.periodo.anio_inicio} – {cfg.periodo.anio_fin}")
print(f"  Meses          : {len(df):,}")
print(f"  Target         : '{cfg.target.nombre}' (binario)")
n_pos = int(df[cfg.target.nombre].sum())
n_neg = len(df) - n_pos
print(f"  Clase 1 (evento)    : {n_pos} meses ({n_pos/len(df):.1%})")
print(f"  Clase 0 (no evento) : {n_neg} meses ({n_neg/len(df):.1%})")
print(f"  Ratio desbalance    : 1 : {n_neg/n_pos:.1f}")
print(f"  Features       : {len(FEATURES)}")
print(f"  Archivo output : data/processed/{cfg.geo.departamento}_deslizamientos_binario_v1.csv")
print("=" * 60)